# PHASE 1
## RAW DATA EXPLORATION — Run this before cleaning
## Step 1: Load each CSV and understand it raw


In [16]:
import pandas as pd
import numpy as np

RAW = r'F:\ecommerce-intelligence\Data\Raw Data' + '\\'

orders      = pd.read_csv(RAW + 'olist_orders_dataset.csv')
customers   = pd.read_csv(RAW + 'olist_customers_dataset.csv')
items       = pd.read_csv(RAW + 'olist_order_items_dataset.csv')
payments    = pd.read_csv(RAW + 'olist_order_payments_dataset.csv')
reviews     = pd.read_csv(RAW + 'olist_order_reviews_dataset.csv')
products    = pd.read_csv(RAW + 'olist_products_dataset.csv')
sellers     = pd.read_csv(RAW + 'olist_sellers_dataset.csv')

## Step 2: shape + head() for every table

In [18]:
datasets = {
    'orders': orders, 'customers': customers, 'items': items,
    'payments': payments, 'reviews': reviews,
    'products': products, 'sellers': sellers
}

for name, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"  TABLE: {name.upper()}  |  Shape: {df.shape}")
    print(f"{'='*60}")
    print(df.head(3).to_string())


  TABLE: ORDERS  |  Shape: (99441, 8)
                           order_id                       customer_id order_status order_purchase_timestamp    order_approved_at order_delivered_carrier_date order_delivered_customer_date order_estimated_delivery_date
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15          2017-10-04 19:55:00           2017-10-10 21:25:13           2017-10-18 00:00:00
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27          2018-07-26 14:31:00           2018-08-07 15:27:45           2018-08-13 00:00:00
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23          2018-08-08 13:50:00           2018-08-17 18:06:29           2018-09-04 00:00:00

  TABLE: CUSTOMERS  |  Shape: (99441, 5)
                        customer_id            

## Step 3: .info for each one.

In [19]:
for name, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"  INFO: {name.upper()}")
    print(f"{'='*60}")
    df.info()


  INFO: ORDERS
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB

  INFO: CUSTOMERS
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  st

## Step 4: describe() for cols.

In [20]:
# shows min/max data catches and bad numbers/values
for name, df in datasets.items():
    numeric_cols = df.select_dtypes(include='number').columns
    if len(numeric_cols) > 0:
        print(f"\n{'='*60}")
        print(f"  DESCRIBE: {name.upper()}")
        print(f"{'='*60}")
        print(df[numeric_cols].describe().round(2).to_string())


  DESCRIBE: CUSTOMERS
       customer_zip_code_prefix
count                  99441.00
mean                   35137.47
std                    29797.94
min                     1003.00
25%                    11347.00
50%                    24416.00
75%                    58900.00
max                    99990.00

  DESCRIBE: ITEMS
       order_item_id     price  freight_value
count      112650.00 112650.00      112650.00
mean            1.20    120.65          19.99
std             0.71    183.63          15.81
min             1.00      0.85           0.00
25%             1.00     39.90          13.08
50%             1.00     74.99          16.26
75%             1.00    134.90          21.15
max            21.00   6735.00         409.68

  DESCRIBE: PAYMENTS
       payment_sequential  payment_installments  payment_value
count           103886.00             103886.00      103886.00
mean                 1.09                  2.85         154.10
std                  0.71                  2.

## Step5: Null audit most imp

In [21]:
print("\nNULL AUDIT ACROSS ALL RAW TABLES\n")
print(f"{'Table':<15} {'Column':<40} {'Nulls':>8} {'Null %':>8}")
print("-" * 75)

for name, df in datasets.items():
    for col in df.columns:
        null_count = df[col].isnull().sum()
        null_pct   = round(null_count / len(df) * 100, 2)
        if null_count > 0:
            print(f"{name:<15} {col:<40} {null_count:>8,} {null_pct:>7.1f}%")


NULL AUDIT ACROSS ALL RAW TABLES

Table           Column                                      Nulls   Null %
---------------------------------------------------------------------------
orders          order_approved_at                             160     0.2%
orders          order_delivered_carrier_date                1,783     1.8%
orders          order_delivered_customer_date               2,965     3.0%
reviews         review_comment_title                       87,656    88.3%
reviews         review_comment_message                     58,247    58.7%
products        product_category_name                         610     1.9%
products        product_name_lenght                           610     1.9%
products        product_description_lenght                    610     1.9%
products        product_photos_qty                            610     1.9%
products        product_weight_g                                2     0.0%
products        product_length_cm                               

## Step 6: Duplicate check

In [22]:
print("\nDUPLICATE CHECK\n")
for name, df in datasets.items():
    dups = df.duplicated().sum()
    print(f"  {name:<15} → {dups:,} duplicate rows")


DUPLICATE CHECK

  orders          → 0 duplicate rows
  customers       → 0 duplicate rows
  items           → 0 duplicate rows
  payments        → 0 duplicate rows
  reviews         → 0 duplicate rows
  products        → 0 duplicate rows
  sellers         → 0 duplicate rows


## Step 7: value Counts on key categorical cols.

In [23]:
print("\n--- orders: order_status value counts ---")
print(orders['order_status'].value_counts())

print("\n--- payments: payment_type value counts ---")
print(payments['payment_type'].value_counts())

print("\n--- reviews: review_score value counts ---")
print(reviews['review_score'].value_counts().sort_index())

print("\n--- customers: top 10 states ---")
print(customers['customer_state'].value_counts().head(10))

print("\n--- products: top 10 categories (Portuguese) ---")
print(products['product_category_name'].value_counts().head(10))


--- orders: order_status value counts ---
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

--- payments: payment_type value counts ---
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

--- reviews: review_score value counts ---
review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

--- customers: top 10 states ---
customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
Name: count, dtype: int64

--- products: top 10 categories (Portuguese) ---
product_category_name
cama_mesa_banho           3029
esporte_lazer             2867
moveis_decoracao          2657
beleza_saude              2444
utilidades_domesticas     2335
a

## Step8: Date range check

In [24]:
orders['order_purchase_timestamp'] = pd.to_datetime(
    orders['order_purchase_timestamp'], errors='coerce'
)

print("\nOrder date range:")
print(f"  Earliest: {orders['order_purchase_timestamp'].min()}")
print(f"  Latest:   {orders['order_purchase_timestamp'].max()}")
print(f"  Null timestamps: {orders['order_purchase_timestamp'].isnull().sum()}")

print("\nDelivery date nulls (expected — cancelled/undelivered orders):")
for col in ['order_approved_at', 'order_delivered_carrier_date',
            'order_delivered_customer_date']:
    raw_nulls = orders[col].isnull().sum()
    print(f"  {col:<40} {raw_nulls:,} nulls")


Order date range:
  Earliest: 2016-09-04 21:15:19
  Latest:   2018-10-17 17:30:18
  Null timestamps: 0

Delivery date nulls (expected — cancelled/undelivered orders):
  order_approved_at                        160 nulls
  order_delivered_carrier_date             1,783 nulls
  order_delivered_customer_date            2,965 nulls


# PHASE 2

## Create a notebook that used for cleaning the raw files.


Step 1:  IMPORTS ALL FILES.

In [1]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv

load_dotenv('../.env')

RAW='../Data/Raw Data/'
PROCESSED='../Data/Processed Data/'

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format','{:.2f}'.format)

## Step 2: Load all csv files from the raw data folder

In [2]:
orders      = pd.read_csv(RAW + 'olist_orders_dataset.csv')
customers   = pd.read_csv(RAW + 'olist_customers_dataset.csv')
items       = pd.read_csv(RAW + 'olist_order_items_dataset.csv')
payments    = pd.read_csv(RAW + 'olist_order_payments_dataset.csv')
reviews     = pd.read_csv(RAW + 'olist_order_reviews_dataset.csv')
products    = pd.read_csv(RAW + 'olist_products_dataset.csv')
sellers     = pd.read_csv(RAW + 'olist_sellers_dataset.csv')
geo         = pd.read_csv(RAW + 'olist_geolocation_dataset.csv')
category_tr = pd.read_csv(RAW + 'product_category_name_translation.csv')

for name, df in{
    'orders': orders,
    'customers': customers,
    'items': items,
    'payments': payments,
    'reviews': reviews,
    'products': products,
    'sellers': sellers,
    'geo': geo,
    'category_tr': category_tr
}.items():
    print(f"{name:15s} -> {df.shape[0]:>7,} rows | {df.shape[1]} cols")
    

orders          ->  99,441 rows | 8 cols
customers       ->  99,441 rows | 5 cols
items           -> 112,650 rows | 7 cols
payments        -> 103,886 rows | 5 cols
reviews         ->  99,224 rows | 7 cols
products        ->  32,951 rows | 9 cols
sellers         ->   3,095 rows | 4 cols
geo             -> 1,000,163 rows | 5 cols
category_tr     ->      71 rows | 2 cols


## Step 3: Initial Data quality audit

In [3]:
def audit(df,name):
    print(f"/{'='*50}")
    print(f" {name}")
    print(f"{'='*50}")
    null_counts=df.isnull().sum()
    null_pct=(null_counts/len(df)*100).round(2)
    audit_df=pd.DataFrame({'nulls':null_counts,'null_%':null_pct})
    print(audit_df[audit_df['nulls']>0])
    print(f"/nDuplicates: {df.duplicated().sum()}")
    print(f"Dtypes:/n{df.dtypes}")
    
audit(orders,'ORDERS')
audit(customers,'CUSTOMERS')
audit(items,'ITEMS')
audit(payments, 'PAYMENTS')
audit(reviews,'REVIEWS')
audit(products,'PRODUCTS')

/==================================================
 ORDERS
                               nulls  null_%
order_approved_at                160    0.16
order_delivered_carrier_date    1783    1.79
order_delivered_customer_date   2965    2.98
/nDuplicates: 0
Dtypes:/norder_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object
/==================================================
 CUSTOMERS
Empty DataFrame
Columns: [nulls, null_%]
Index: []
/nDuplicates: 0
Dtypes:/ncustomer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object
/==================================================
 ITEMS
Empty DataFrame
Columns: [nulls, null_%]
In

## step 4: fix datetime cols

In [4]:
date_cols_orders = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols_orders:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

reviews['review_creation_date']    = pd.to_datetime(reviews['review_creation_date'], errors='coerce')
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'], errors='coerce')
items['shipping_limit_date']       = pd.to_datetime(items['shipping_limit_date'], errors='coerce')

print("Date columns converted.")
print(orders[date_cols_orders].dtypes)

Date columns converted.
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


## Step 5: Clean ORDERS

In [5]:
print(f"Orders before: {len(orders)}")

# Keep only delivered orders for most analysis (but keep all for status analysis)
orders_clean = orders.copy()

# Drop rows where purchase timestamp is null (data entry error)
orders_clean = orders_clean.dropna(subset=['order_purchase_timestamp'])

# Fill approved_at where null with purchase + 30 min (estimated approval)
mask_approved = orders_clean['order_approved_at'].isnull()
orders_clean.loc[mask_approved, 'order_approved_at'] = (
    orders_clean.loc[mask_approved, 'order_purchase_timestamp'] + pd.Timedelta(minutes=30)
)

print(f"Orders after cleaning: {len(orders_clean)}")
print(f"Null delivered dates (expected for non-delivered): {orders_clean['order_delivered_customer_date'].isnull().sum()}")

Orders before: 99441
Orders after cleaning: 99441
Null delivered dates (expected for non-delivered): 2965


## Step6: Clean PRODUCTS that means clean null value and convert portuguese to english

In [6]:
print(f"Products before: {len(products)}")
print(f"Null category names: {products['product_category_name'].isnull().sum()}")

# Merge English translation
products_clean = products.merge(category_tr, on='product_category_name', how='left')

# Fill null category with 'uncategorized'
products_clean['product_category_name_english'] = (
    products_clean['product_category_name_english']
    .fillna('uncategorized')
)

# Fill numeric nulls with column median
num_cols = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
for col in num_cols:
    median_val = products_clean[col].median()
    null_count = products_clean[col].isnull().sum()
    products_clean[col] = products_clean[col].fillna(median_val)
    print(f"  Filled {null_count} nulls in {col} with median={median_val:.1f}")

# Fill product_name_lenght / description nulls with 0
products_clean['product_name_lenght']        = products_clean['product_name_lenght'].fillna(0)
products_clean['product_description_lenght'] = products_clean['product_description_lenght'].fillna(0)
products_clean['product_photos_qty']         = products_clean['product_photos_qty'].fillna(0)

print(f"Products after: {len(products_clean)}")

Products before: 32951
Null category names: 610
  Filled 2 nulls in product_weight_g with median=700.0
  Filled 2 nulls in product_length_cm with median=25.0
  Filled 2 nulls in product_height_cm with median=13.0
  Filled 2 nulls in product_width_cm with median=20.0
Products after: 32951


## Step 7: Clean REVIEWS

In [7]:
print(f"Reviews before: {len(reviews)}")
print(f"Null review scores: {reviews['review_score'].isnull().sum()}")

reviews_clean = reviews.copy()

# Keep only 1 review per order (take max score if duplicates)
reviews_clean = (
    reviews_clean
    .sort_values('review_score', ascending=False)
    .drop_duplicates(subset='order_id', keep='first')
)

# review_score nulls — flag as -1 (unknown), NOT fill with mean
# We want to distinguish "no review given" from "low score"
reviews_clean['review_score'] = reviews_clean['review_score'].fillna(-1)
reviews_clean['has_review']   = (reviews_clean['review_score'] > 0).astype(int)

print(f"Reviews after dedup: {len(reviews_clean)}")

Reviews before: 99224
Null review scores: 0
Reviews after dedup: 98673


## Step 8: Aggregate PAYMENTS

In [8]:
# Payments has multiple rows per order (split payments, installments)
payments_agg = payments.groupby('order_id').agg(
    total_payment_value = ('payment_value', 'sum'),
    payment_installments = ('payment_installments', 'max'),
    payment_types_used   = ('payment_type', lambda x: '|'.join(sorted(x.unique())))
).reset_index()

# Primary payment type (most common one for that order)
primary_payment = (
    payments
    .groupby('order_id')['payment_type']
    .agg(lambda x: x.value_counts().index[0])
    .reset_index()
    .rename(columns={'payment_type': 'primary_payment_type'})
)

payments_agg = payments_agg.merge(primary_payment, on='order_id', how='left')
print(f"Payments aggregated: {payments_agg.shape}")
print(payments_agg.head(3))

Payments aggregated: (99440, 5)
                           order_id  total_payment_value  \
0  00010242fe8c5a6d1ba2dd792cb16214                72.19   
1  00018f77f2f0320c557190d7a144bdd3               259.83   
2  000229ec398224ef6ca0657da4fc703e               216.87   

   payment_installments payment_types_used primary_payment_type  
0                     2        credit_card          credit_card  
1                     3        credit_card          credit_card  
2                     5        credit_card          credit_card  


## Step9: Aggregate ITEMS one row per order

In [9]:
# ============================================================
items_agg = items.groupby('order_id').agg(
    item_count          = ('order_item_id', 'count'),
    total_freight_value = ('freight_value', 'sum'),
    total_price         = ('price', 'sum'),
    unique_sellers      = ('seller_id', 'nunique')
).reset_index()

# Keep product_id of most expensive item (for category join)
most_expensive_item = (
    items
    .sort_values('price', ascending=False)
    .drop_duplicates(subset='order_id', keep='first')[['order_id', 'product_id', 'seller_id']]
)

items_agg = items_agg.merge(most_expensive_item, on='order_id', how='left')
print(f"Items aggregated: {items_agg.shape}")

Items aggregated: (98666, 7)


## Step 10: Build MASTER TABLE( big merge all tables)

In [10]:
master = (
    orders_clean
    .merge(customers,      on='customer_id',  how='left')
    .merge(items_agg,      on='order_id',     how='left')
    .merge(payments_agg,   on='order_id',     how='left')
    .merge(reviews_clean[['order_id', 'review_score', 'has_review']], on='order_id', how='left')
    .merge(products_clean[['product_id', 'product_category_name_english',
                            'product_weight_g']], on='product_id', how='left')
    .merge(sellers[['seller_id', 'seller_city', 'seller_state']], on='seller_id', how='left')
)

print(f"Master table shape: {master.shape}")
print(master.columns.tolist())

Master table shape: (99441, 28)
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'item_count', 'total_freight_value', 'total_price', 'unique_sellers', 'product_id', 'seller_id', 'total_payment_value', 'payment_installments', 'payment_types_used', 'primary_payment_type', 'review_score', 'has_review', 'product_category_name_english', 'product_weight_g', 'seller_city', 'seller_state']


## Step 11: Engineering Step new columns create by previous.

In [ ]:
# 1. Delivery delay in days (negative = early, positive = late)
delivered_mask = master['order_delivered_customer_date'].notna()
master['delivery_delay_days'] = np.nan

master.loc[delivered_mask, 'delivery_delay_days'] = (
    (master.loc[delivered_mask, 'order_delivered_customer_date'] -
master.loc[delivered_mask, 'order_estimated_delivery_date'])
    .dt.days
)

# 2. Actual delivery days (purchase → delivered)
master.loc[delivered_mask, 'actual_delivery_days'] = (
    (master.loc[delivered_mask, 'order_delivered_customer_date'] -
master.loc[delivered_mask, 'order_purchase_timestamp'])
    .dt.days
)

# 3. Order value bucket (total_payment_value)
def value_bucket(val):
    if pd.isna(val):    return 'unknown'
    elif val < 100:     return 'low'
    elif val < 500:     return 'mid'
    else:               return 'high'

master['order_value_bucket'] = master['total_payment_value'].apply(value_bucket)

# 4. Time dimensions (for Power BI slicers)
master['order_year']    = master['order_purchase_timestamp'].dt.year
master['order_month']   = master['order_purchase_timestamp'].dt.month
master['order_quarter'] = master['order_purchase_timestamp'].dt.quarter
master['order_month_name'] = master['order_purchase_timestamp'].dt.strftime('%b')
master['order_yearmonth']  = master['order_purchase_timestamp'].dt.to_period('M').astype(str)

# 5. On-time delivery flag
master['is_on_time'] = (master['delivery_delay_days'] <= 0).astype('Int64')

# 6. Total order value (price + freight)
master['order_total_with_freight'] = master['total_price'] + master['total_freight_value']

# 7. Review category
def review_category(score):
    if score == -1:   return 'no_review'
    elif score <= 2:  return 'negative'
    elif score == 3:  return 'neutral'
    else:             return 'positive'

master['review_category'] = master['review_score'].apply(review_category)

print("Derived columns added:")
print(master[['delivery_delay_days', 'actual_delivery_days', 'order_value_bucket',
'order_year', 'order_month', 'is_on_time', 'review_category']].head(5))

Derived columns added:
   delivery_delay_days  actual_delivery_days order_value_bucket  order_year  \
0                -8.00                  8.00                low        2017   
1                -6.00                 13.00                mid        2018   
2               -18.00                  9.00                mid        2018   
3               -13.00                 13.00                low        2017   
4               -10.00                  2.00                low        2018   

   order_month  is_on_time review_category  
0           10           1        positive  
1            7           1        positive  
2            8           1        positive  
3           11           1        positive  
4            2           1        positive  


## Step 12: Final null audit on master

In [13]:
null_summary = master.isnull().sum()
null_pct = (null_summary / len(master) * 100).round(2)
final_audit = pd.DataFrame({'nulls': null_summary, 'null_%': null_pct})
print(final_audit[final_audit['nulls'] > 0].sort_values('null_%', ascending=False))

                               nulls  null_%
order_delivered_customer_date   2965    2.98
actual_delivery_days            2965    2.98
delivery_delay_days             2965    2.98
order_delivered_carrier_date    1783    1.79
item_count                       775    0.78
total_price                      775    0.78
total_freight_value              775    0.78
unique_sellers                   775    0.78
product_id                       775    0.78
product_weight_g                 775    0.78
product_category_name_english    775    0.78
seller_id                        775    0.78
seller_state                     775    0.78
order_total_with_freight         775    0.78
seller_city                      775    0.78
review_score                     768    0.77
has_review                       768    0.77
primary_payment_type               1    0.00
payment_types_used                 1    0.00
payment_installments               1    0.00
total_payment_value                1    0.00


## Step 13: Export cleaned master to csv

In [14]:
master.to_csv(PROCESSED + 'master_orders.csv', index=False)
print(f"Saved: {PROCESSED}master_orders.csv")
print(f"Shape: {master.shape}")
print(f"Columns: {master.columns.tolist()}")

Saved: ../Data/Processed Data/master_orders.csv
Shape: (99441, 39)
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'item_count', 'total_freight_value', 'total_price', 'unique_sellers', 'product_id', 'seller_id', 'total_payment_value', 'payment_installments', 'payment_types_used', 'primary_payment_type', 'review_score', 'has_review', 'product_category_name_english', 'product_weight_g', 'seller_city', 'seller_state', 'delivery_delay_days', 'actual_delivery_days', 'order_value_bucket', 'order_year', 'order_month', 'order_quarter', 'order_month_name', 'order_yearmonth', 'is_on_time', 'order_total_with_freight', 'review_category']


## Step 14: Export profiling workbook to Excel

In [15]:
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows

wb = Workbook()

# ---- Sheet 1: Data Profile ----
ws1 = wb.active
ws1.title = "Data Profile"

profile_data = []
for col in master.columns:
    profile_data.append({
        'Column': col,
        'Dtype': str(master[col].dtype),
        'Non_Null': master[col].notna().sum(),
        'Null_Count': master[col].isnull().sum(),
        'Null_Pct': round(master[col].isnull().mean() * 100, 2),
        'Unique_Values': master[col].nunique(),
        'Sample': str(master[col].dropna().iloc[0]) if master[col].notna().any() else 'N/A'
    })

profile_df = pd.DataFrame(profile_data)

header_fill = PatternFill("solid", fgColor="1F4E79")
header_font = Font(color="FFFFFF", bold=True)

for r_idx, row in enumerate(dataframe_to_rows(profile_df, index=False, header=True), 1):
    ws1.append(row)
    if r_idx == 1:
        for cell in ws1[r_idx]:
            cell.fill = header_fill
            cell.font = header_font

ws1.column_dimensions['A'].width = 40
ws1.column_dimensions['G'].width = 30

# ---- Sheet 2: Revenue by State ----
ws2 = wb.create_sheet("Revenue by State")
state_rev = (
    master.groupby('customer_state')['total_payment_value']
    .agg(['sum', 'count', 'mean'])
    .round(2)
    .reset_index()
    .rename(columns={'sum': 'total_revenue', 'count': 'order_count', 'mean': 'avg_order_value'})
    .sort_values('total_revenue', ascending=False)
)
for row in dataframe_to_rows(state_rev, index=False, header=True):
    ws2.append(row)

# ---- Sheet 3: Monthly Revenue ----
ws3 = wb.create_sheet("Monthly Revenue")
monthly = (
    master.groupby(['order_year', 'order_month', 'order_yearmonth'])['total_payment_value']
    .sum().round(2).reset_index()
    .sort_values(['order_year', 'order_month'])
)
for row in dataframe_to_rows(monthly, index=False, header=True):
    ws3.append(row)

# ---- Sheet 4: Category Performance ----
ws4 = wb.create_sheet("Category Performance")
cat_perf = (
    master.groupby('product_category_name_english').agg(
        total_revenue  = ('total_payment_value', 'sum'),
        order_count    = ('order_id', 'count'),
        avg_review     = ('review_score', lambda x: x[x > 0].mean()),
        avg_delay_days = ('delivery_delay_days', 'mean')
    ).round(2).reset_index()
    .sort_values('total_revenue', ascending=False)
)
for row in dataframe_to_rows(cat_perf, index=False, header=True):
    ws4.append(row)

# ---- Sheet 5: Payment Analysis ----
ws5 = wb.create_sheet("Payment Analysis")
pay_analysis = (
    master.groupby('primary_payment_type').agg(
        order_count    = ('order_id', 'count'),
        total_revenue  = ('total_payment_value', 'sum'),
        avg_order_value= ('total_payment_value', 'mean'),
        avg_installments=('payment_installments', 'mean')
    ).round(2).reset_index()
)
for row in dataframe_to_rows(pay_analysis, index=False, header=True):
    ws5.append(row)

wb.save(PROCESSED + 'ecommerce_profiling.xlsx')
print("Excel profiling workbook saved.")

Excel profiling workbook saved.
